In [1]:
import json
import random
from pathlib import Path
from IPython.display import Markdown, display


# =========================
# CONFIG
# =========================

CLEAN_JSON_PATH = Path(
    "/home/rpinter/art-in-swh/artworks/PROMPT_10f_20260603/clean_artworks_label_prediction.json"
)

SRC_ROOT_DIR = Path(
    "/home/rpinter/links/projects/def-baudry/shared/data/artworks/src"
)


# =========================
# HELPERS
# =========================

def read_json(file_path: Path):
    with file_path.open("r", encoding="utf-8") as f:
        return json.load(f)


def normalize_path(path: str) -> str:
    """
    Convert absolute artwork paths to paths relative to SRC_ROOT_DIR.
    """
    path = str(path)

    marker = "/artworks/src/"
    if marker in path:
        return path.split(marker, 1)[-1].lstrip("/")

    return path.lstrip("/")


def ensure_list(value):
    if value is None:
        return []

    if isinstance(value, list):
        return value

    return [value]


def flatten_labels(item: dict) -> set[str]:
    """
    Convert predicted_labels into one flat set of tags.
    """
    labels = item.get("predicted_labels", {})

    tags = set()
    tags.update(ensure_list(labels.get("entities", [])))
    tags.update(ensure_list(labels.get("interaction", [])))
    tags.update(ensure_list(labels.get("outcome", [])))

    return tags


def print_code_file(file_path, language="javascript"):
    """
    Print a source-code file as a Markdown code block in Jupyter.
    """
    file_path = Path(file_path)
    code = file_path.read_text(encoding="utf-8", errors="ignore")

    display(Markdown(f"```{language}\n{code}\n```"))


def print_sample_code(sample, language="javascript"):
    """
    Print the sample tags/labels and then the source code.
    """
    print("File:")
    print(sample["file_path"])

    # print("\nPredicted labels:")
    # print(json.dumps(sample["predicted_labels"], indent=2, ensure_ascii=False))

    print("\nAll tags:")
    print(sample["all_tags"])

    print("\nSource code:")
    file_path = SRC_ROOT_DIR / sample["file_path"]
    print_code_file(file_path, language=language)


def get_random_src_by_tags(
    tags: list[str],
    k: int = 10,
    exact: bool = False,
    clean_json_path: Path | None = None,
    src_root_dir: Path | None = None,
    seed: int = 42,
) -> list[dict]:
    """
    Find k random source-code files from clean_artworks_label_prediction.json.

    exact=False:
        returns files that contain at least all tags passed.

    exact=True:
        returns files whose full label set is exactly equal to the tags passed.

    Excludes files containing "Daniel Shiffman".
    """
    if clean_json_path is None:
        clean_json_path = CLEAN_JSON_PATH

    if src_root_dir is None:
        src_root_dir = SRC_ROOT_DIR

    data = read_json(clean_json_path)

    target_tags = set(tags)
    matches = []

    for item in data:
        item_tags = flatten_labels(item)

        if exact:
            keep = item_tags == target_tags
        else:
            keep = target_tags.issubset(item_tags)

        if not keep:
            continue

        relative_file_path = normalize_path(item["file_path"])
        src_path = src_root_dir / relative_file_path

        if not src_path.exists():
            continue

        src_code = src_path.read_text(encoding="utf-8", errors="ignore")

        if "Daniel Shiffman" in src_code:
            continue

        matches.append({
            "file_path": relative_file_path,
            "predicted_labels": item.get("predicted_labels", {}),
            "label_combination": item.get("label_combination", ""),
            "all_tags": sorted(item_tags),
        })

    rng = random.Random(seed)
    return rng.sample(matches, min(k, len(matches)))


def show_samples(samples: list[dict]):
    """
    Print sample paths and labels, without printing source code.
    """
    for i, sample in enumerate(samples, start=1):
        print("=" * 100)
        print(f"Sample {i}")
        print(f"File: {sample['file_path']}")
        print("Labels:")
        print(json.dumps(sample["predicted_labels"], indent=2, ensure_ascii=False))


In [2]:
tags = [
    # entities
    # "processed_audio",
    # "processed_image",
    # "processed_text",
    # "synthesized_sound",
    # "synthesized_text",
    "synthesized_image",
    # "randomness",

    # interaction
    "yes",
    # "no",

    # outcome
    "visual",
    # "auditory",
    "static",
    # "time_based",
]

samples = get_random_src_by_tags(
    tags=tags,
    k=100,
    exact=True,
    # exact=False,
)
i = 1
print_sample_code(samples[i])

File:
tezzutezzu/p5.js/test/manual-test-examples/dom/checkbox_test/sketch.js

All tags:
['static', 'synthesized_image', 'visual', 'yes']

Source code:


```javascript
var checkbox;
var testcheck;

function setup() {
  checkbox = createCheckbox('the label');
  checkbox.value('some value');

  // What should this be called??
  // it's wrapping 'onchange'
  checkbox.changed(myCheckedEvent); // even for when the user does something

  testcheck = select('#checktest');
  testcheck.changed(myCheckedEvent);
}

function draw() {
  background(0);
  // No argument return its state
  if (checkbox.checked()) {
    background(255, 0, 0);
  }
  if (testcheck.checked()) {
    background(255, 0, 255);
  }
}

function myCheckedEvent() {
  if (this.checked()) {
    console.log(this.value() + ' is checked!');
  } else {
    console.log(this.value() + ' is not checked!');
  }
}

```